In [ ]:

!pip install -q --upgrade bitsandbytes accelerate

In [ ]:
from ai_tools.hugging_face import HuggingFaceQuery

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
SYSTEM_PROMPT = """### ROLE
You are the **Mock Data Generator Assistant**, a specialized AI designed to generate high-fidelity, domain-specific mock datasets based on vague or specific business problem statements.

### OBJECTIVE
Your goal is to interpret a user's business scenario (e.g., "HR attrition analysis," "Supply chain logistics," "E-commerce transaction logs") and generate a realistic dataset in **strict JSON format**.

### OPERATIONAL RULES

1.  **Analyze the Domain:** deeply understand the industry implied by the prompt. If the user asks for "hospital patients," include medical-specific fields like `diagnosis_code`, `admission_date`, `insurance_provider`, and `blood_type`.
2.  **Infer Attributes:** Do not wait for the user to list columns. You must infer the most valuable 10-15 attributes that a data scientist or developer would need to solve the specific business problem.
3.  **Enforce Realism:**
    * Do not use placeholder values like "User1", "Test Data", or "asdf".
    * Use realistic names, addresses, UUIDs, timestamps, and domain-specific jargon (e.g., SKU numbers, ICD-10 codes, Stock tickers).
    * Ensure logical consistency (e.g., `delivery_date` must be after `order_date`; `age` must match `date_of_birth`).
4.  **Volume:** Unless specified otherwise, generate **10 records** to provide a representative sample.

### FORMATTING CONSTRAINTS (CRITICAL)
* **Output:** You must return **ONLY** a valid JSON array of objects.
* **No Chatter:** Do not provide introductions, explanations, or "Here is your data" text. Start with `[` and end with `]`.
* **Syntax:** Ensure strict JSON compliance (double quotes for keys/strings, no trailing commas).
* **JSON Layout:** Always return an array of objects, never a single object.

### ERROR HANDLING
If the user request is gibberish or unrelated to data generation, return a JSON object with a single error field:
`[{"error": "Unable to generate data. Please provide a valid business scenario."}]`

### EXAMPLES

**User Input:** "I need to test a fraud detection system for credit cards."
**System Output:**
[
  {
    "transaction_id": "550e8400-e29b-41d4-a716-446655440000",
    "timestamp": "2023-10-27T14:30:00Z",
    "amount": 1250.00,
    "currency": "USD",
    "merchant": "Global Electronics",
    "merchant_category_code": 5732,
    "card_holder": "Jane Doe",
    "card_last_four": "4242",
    "device_ip": "192.168.1.1",
    "is_flagged": true,
    "risk_score": 0.89
  },
  {
    ...
  }
]"""

**OPEN SOURCE MODEL**

In [ ]:
mock_data_generator = HuggingFaceQuery(
    system_prompt=SYSTEM_PROMPT
)

In [5]:
output_data = mock_data_generator.query("I need Data to test my IFRS9 Accounting.", max_new_tokens=10000)

Loading tokenizer for meta-llama/Llama-3.2-3B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading model meta-llama/Llama-3.2-3B-Instruct on cuda with 4bit quantization...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
def extract_json_segment(text: str) -> str:
    """
    Finds the first occurrence of "json", then extracts the content 
    starting from the first "[" after it, up to the last "]".
    """
    # Find the "json" keyword
    json_keyword_index = text.find("json")
    
    if json_keyword_index != -1:
        # Find the first '[' after the "json" keyword
        start_index = text.find("[", json_keyword_index)
        end_index = text.rfind("]")
        
        # Ensure proper order and existence
        if start_index != -1 and end_index != -1 and start_index < end_index:
            return text[start_index : end_index + 1]
            
    return ""

In [9]:
from ai_tools.tools import pretty_print_json
pretty_print_json(extract_json_segment(output_data))

```json
[
  {
    "accounting_date": "2022-01-01",
    "account_id": "123456789",
    "account_name": "Inventory",
    "asset_class": "Current",
    "depreciation_method": "Straight Line",
    "depreciation_rate": 0.1,
    "cost": 100000.0,
    "original_cost": 120000.0,
    "amortization_expense": 1000.0,
    " impairment_loss": 0.0,
    "revaluation_gain": 0.0,
    "revaluation_loss": 0.0
  },
  {
    "accounting_date": "2022-06-30",
    "account_id": "987654321",
    "account_name": "Deposits",
    "asset_class": "Non-Current",
    "depreciation_method": "None",
    "depreciation_rate": 0.0,
    "cost": 50000.0,
    "original_cost": 50000.0,
    "amortization_expense": 0.0,
    "impairment_loss": 0.0,
    "revaluation_gain": 0.0,
    "revaluation_loss": 0.0
  },
  {
    "accounting_date": "2022-03-31",
    "account_id": "555555555",
    "account_name": "Loans Receivable",
    "asset_class": "Non-Current",
    "depreciation_method": "None",
    "depreciation_rate": 0.0,
    "cost": 200000.0,
    "original_cost": 200000.0,
    "amortization_expense": 1000.0,
    "impairment_loss": 0.0,
    "revaluation_gain": 0.0,
    "revaluation_loss": 0.0
  },
  {
    "accounting_date": "2022-12-31",
    "account_id": "111111111",
    "account_name": "Prepaid Expenses",
    "asset_class": "Current",
    "depreciation_method": "Straight Line",
    "depreciation_rate": 0.1,
    "cost": 30000.0,
    "original_cost": 30000.0,
    "amortization_expense": 250.0,
    "impairment_loss": 0.0,
    "revaluation_gain": 0.0,
    "revaluation_loss": 0.0
  },
  {
    "accounting_date": "2022-09-30",
    "account_id": "222222222",
    "account_name": "Investments",
    "asset_class": "Non-Current",
    "depreciation_method": "None",
    "depreciation_rate": 0.0,
    "cost": 150000.0,
    "original_cost": 150000.0,
    "amortization_expense": 0.0,
    "impairment_loss": 0.0,
    "revaluation_gain": 0.0,
    "revaluation_loss": 0.0
  }
]
```

**FRONTIER MODELS**

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools.tools import LLMQuery

In [3]:
mock_data_generator_frontier = LLMQuery(
    system_prompt= SYSTEM_PROMPT,
    json_format=True
)

In [ ]:
import gradio as gr
from ai_tools.tools import MODEL_DICT
import os
import pandas as pd

# Flatten model list for the dropdown
all_models = []
for models in MODEL_DICT.values():
    all_models.extend(list(models))
all_models.sort()

def generate_data(user_input, model_name):
    """
    Generates mock data based on user input and selected model.
    Returns a dataframe and a path to the CSV file.
    """
    if not user_input:
        return None, None

    try:
        # Query the LLM
        response = mock_data_generator_frontier.query(
            user_input, model=model_name, display_output=False, use_history=False
        )

  
        df = pd.read_json(StringIO(response))

        # Save to CSV for download
        # We save to a temporary file or a static one;
        # overwriting "generated_mock_data.csv" is fine for single user demo
        csv_filename = os.path.abspath(os.path.join(
            os.getcwd(), "mock_data.csv"
        ))
        df.to_csv(csv_filename, index=False)

        return df, csv_filename

    except Exception as e:
        # Return an error dataframe
        error_df = pd.DataFrame({"Error": [f"Failed to generate data: {str(e)}"]})
        return error_df, None


# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎲 Frontier Models - Mock Data Generator")
    gr.Markdown(
        "Describe the data you need, select a model, and generate a downloadable dataset."
    )

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=all_models,
                value="gemini-flash-latest",
                label="Select Model",
                info="Choose the LLM to generate your data.",
                interactive=True,
            )
        with gr.Column(scale=3):
            pass  # Spacer

    user_input = gr.Textbox(
        label="Data Description",
        placeholder="e.g., I need a dataset of 10 fake transactions for a fraud detection system by extracting json...",
        lines=2,
    )

    generate_btn = gr.Button("🚀 Generate Data", variant="primary")

    # DataFrame output - supports copying/sorting
    output_df = gr.DataFrame(label="Generated Data", interactive=False, wrap=True)

    with gr.Row():
        download_btn = gr.DownloadButton("📥 Download CSV", label="Download CSV")

    # Event wiring
    # Click button
    generate_btn.click(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

    # Press Enter (submit)
    # The default behavior of submit is to trigger the function but NOT clear the input unless we ask it to.
    user_input.submit(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

demo.launch(inbrowser=True)

ModuleNotFoundError: No module named 'ai_tools'